# MC Dropout for Uncertainty Estimation on MNIST

Reimplementation of the practical core of **Gal & Ghahramani (2016), *Dropout as a Bayesian Approximation***.

This notebook is a thin runner: it calls the scripts in `experiments/` (which use the reusable
`src/mc_dropout` package) and displays the figures and tables they produce. That keeps the notebook
and the command-line results identical.


## 0. Setup

**Google Colab:** set the runtime to GPU, then clone the repo and move into it:

```python
# !git clone <your-repo-url>.git
# %cd mc-dropout-mnist
# !pip -q install -r requirements.txt
```

**Local:** just run from the repository root after `pip install -r requirements.txt`.


In [ ]:
import os, pandas as pd
from IPython.display import Image, display

# Run all cells from the repository root so the relative paths below resolve.
assert os.path.isdir('experiments') and os.path.isdir('src'), \
    'Run this notebook from the repository root (the folder containing experiments/ and src/).'
print('Working dir:', os.getcwd())


## 1. Train the model

Trains the dropout CNN and saves `results/model.pt` plus the training curves.


In [ ]:
!python experiments/01_train.py --epochs 5


In [ ]:
display(Image('figures/training_curves.png'))


## 2. Standard vs MC Dropout + uncertainty of correct/wrong predictions


In [ ]:
!python experiments/02_uncertainty.py --T 30


In [ ]:
display(pd.read_csv('results/summary.csv'))
display(Image('figures/entropy_correct_vs_wrong.png'))
display(Image('figures/variance_correct_vs_wrong.png'))
display(Image('figures/confident_examples.png'))
display(Image('figures/uncertain_examples.png'))


## 3. Rotating-digit experiment (the paper's signature figure)


In [ ]:
!python experiments/03_rotating_digit.py --digit 7 --T 50


In [ ]:
display(Image('figures/rotating_digit.png'))


## 4. Out-of-distribution detection (MNIST vs FashionMNIST)


In [ ]:
!python experiments/04_ood.py --T 30


In [ ]:
display(pd.read_csv('results/ood_summary.csv'))
display(Image('figures/ood_entropy_hist.png'))


## 5. How accuracy / NLL converge with the number of MC samples T


In [ ]:
!python experiments/05_accuracy_vs_T.py --max-T 100


In [ ]:
display(Image('figures/accuracy_vs_T.png'))


## 6. Multi-seed statistics (optional, slower — retrains a few models)


In [ ]:
!python experiments/06_multiseed.py --seeds 0 1 2 --epochs 3


In [ ]:
display(pd.read_csv('results/multiseed_stats.csv'))


## Interpretation

- Standard and MC-Dropout accuracy are essentially identical: MC Dropout adds **uncertainty**, not raw accuracy.
- Wrong predictions carry much higher predictive entropy than correct ones.
- Rotating a digit drives entropy up as the shape becomes ambiguous.
- FashionMNIST (OOD) gets systematically higher entropy than MNIST, giving strong OOD-detection AUROC.
- The MC estimate stabilises after roughly 20-50 forward passes.
